# JetGuard - Feature Engineering

This notebook prepares leakage-free tabular features from the NASA C-MAPSS FD001 dataset.

The steps are:
- load the processed training data and raw test data
- remove constant sensors
- create rolling statistics and lag features
- split engines into 80% training and 20% validation groups
- save processed feature datasets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
VALIDATION_FRACTION = 0.20
ROLLING_WINDOWS = [5, 10, 30]
LAG_STEPS = [1, 5, 10]

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\ahzam\Desktop\Projects\JetGuard


In [2]:
columns = (
    ["engine_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(PROCESSED_DIR / "train_FD001_processed.csv")
test_df = pd.read_csv(
    DATASET_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns,
)
test_rul = pd.read_csv(
    DATASET_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL_true"],
)["RUL_true"]

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Test RUL shape:", test_rul.shape)

Training shape: (20631, 28)
Test shape: (13096, 26)
Test RUL shape: (100,)


In [3]:
sensor_columns = [column for column in columns if column.startswith("sensor_")]

constant_sensors = [
    sensor
    for sensor in sensor_columns
    if train_df[sensor].nunique(dropna=False) == 1
]
informative_sensors = [
    sensor for sensor in sensor_columns if sensor not in constant_sensors
]

print("Constant sensors:", constant_sensors)
print("Informative sensors:", informative_sensors)
print("Informative sensor count:", len(informative_sensors))

Constant sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Informative sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
Informative sensor count: 15


In [4]:
def add_engine_features(frame, sensors):
    """Create within-engine current, rolling, and lag features without future leakage."""
    result = frame.sort_values(["engine_id", "cycle"]).copy()
    feature_columns = ["engine_id", "cycle", "setting_1", "setting_2", "setting_3"]

    for window in ROLLING_WINDOWS:
        for sensor in sensors:
            mean_column = f"{sensor}_roll_mean_{window}"
            std_column = f"{sensor}_roll_std_{window}"
            grouped = result.groupby("engine_id", sort=False)[sensor]
            result[mean_column] = grouped.transform(
                lambda values, size=window: values.rolling(size, min_periods=1).mean()
            )
            result[std_column] = grouped.transform(
                lambda values, size=window: values.rolling(size, min_periods=2).std()
            ).fillna(0.0)
            feature_columns.extend([mean_column, std_column])

    for lag in LAG_STEPS:
        for sensor in sensors:
            lag_column = f"{sensor}_lag_{lag}"
            result[lag_column] = result.groupby("engine_id", sort=False)[sensor].shift(lag)
            result[lag_column] = result[lag_column].fillna(result[sensor])
            feature_columns.append(lag_column)

    feature_columns.extend(sensors)
    return result, feature_columns


train_feature_frame, engineered_columns = add_engine_features(
    train_df, informative_sensors
)

# The first 30-cycle window can be generated only after 30 historical cycles.
train_feature_frame = train_feature_frame[
    train_feature_frame["cycle"] >= max(ROLLING_WINDOWS)
].copy()

print("Engineered training shape:", train_feature_frame.shape)
print("Engineered feature count:", len(engineered_columns))

Engineered training shape: (17731, 163)
Engineered feature count: 155


C:\Users\ahzam\AppData\Local\Temp\ipykernel_22284\1333232178.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[lag_column] = result.groupby("engine_id", sort=False)[sensor].shift(lag)


In [5]:
engine_ids = np.sort(train_feature_frame["engine_id"].unique())
rng = np.random.default_rng(RANDOM_SEED)
shuffled_engines = rng.permutation(engine_ids)
validation_size = int(round(len(shuffled_engines) * VALIDATION_FRACTION))
validation_engines = np.sort(shuffled_engines[:validation_size])
training_engines = np.sort(shuffled_engines[validation_size:])

train_features = train_feature_frame[
    train_feature_frame["engine_id"].isin(training_engines)
].copy()
validation_features = train_feature_frame[
    train_feature_frame["engine_id"].isin(validation_engines)
].copy()

print("Training engines:", len(training_engines))
print("Validation engines:", len(validation_engines))
print("Training rows:", len(train_features))
print("Validation rows:", len(validation_features))

assert set(training_engines).isdisjoint(set(validation_engines))
assert not set(train_features["engine_id"]) & set(validation_features["engine_id"])

Training engines: 80
Validation engines: 20
Training rows: 14459
Validation rows: 3272


In [6]:
test_feature_frame, test_columns = add_engine_features(test_df, informative_sensors)

# For each test engine, retain only the final observed 30-cycle context.
rul_by_engine = dict(zip(sorted(test_feature_frame["engine_id"].unique()), test_rul))
test_feature_frame = test_feature_frame.groupby("engine_id", sort=False).tail(1).copy()
test_feature_frame["RUL_true"] = test_feature_frame["engine_id"].map(rul_by_engine)

print("Test feature shape:", test_feature_frame.shape)
test_feature_frame[["engine_id", "cycle", "RUL_true"]].head(10)

Test feature shape: (100, 162)


C:\Users\ahzam\AppData\Local\Temp\ipykernel_22284\1333232178.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[lag_column] = result.groupby("engine_id", sort=False)[sensor].shift(lag)


,engine_id,cycle,RUL_true
30,1,31,112
79,2,49,98
205,3,126,69
311,4,106,82
409,5,98,91
514,6,105,93
674,7,160,91
840,8,166,95
895,9,55,111
1087,10,192,96


In [7]:
# Scaling is fitted on training engines only and can be reused for validation/test data.
numeric_features = [
    column
    for column in engineered_columns
    if column not in {"engine_id", "cycle"}
]

scaler = StandardScaler()
scaler.fit(train_features[numeric_features])

train_features_scaled = train_features.copy()
validation_features_scaled = validation_features.copy()
test_features_scaled = test_feature_frame.copy()

train_features_scaled[numeric_features] = scaler.transform(train_features[numeric_features])
validation_features_scaled[numeric_features] = scaler.transform(validation_features[numeric_features])
test_features_scaled[numeric_features] = scaler.transform(test_feature_frame[numeric_features])

print("Scaling fit complete")

Scaling fit complete


In [8]:
target_columns = ["RUL", "RUL_capped"]

train_features_scaled.to_csv(
    PROCESSED_DIR / "train_FD001_features.csv", index=False
)
validation_features_scaled.to_csv(
    PROCESSED_DIR / "validation_FD001_features.csv", index=False
)
test_features_scaled.to_csv(
    PROCESSED_DIR / "test_FD001_features.csv", index=False
)

metadata = {
    "dataset": "FD001",
    "random_seed": RANDOM_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "training_engines": training_engines.tolist(),
    "validation_engines": validation_engines.tolist(),
    "constant_sensors": constant_sensors,
    "informative_sensors": informative_sensors,
    "rolling_windows": ROLLING_WINDOWS,
    "lag_steps": LAG_STEPS,
    "numeric_features": numeric_features,
    "targets": target_columns,
    "test_target": "RUL_true",
}

import json
(PROCESSED_DIR / "feature_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("Saved:")
print(PROCESSED_DIR / "train_FD001_features.csv")
print(PROCESSED_DIR / "validation_FD001_features.csv")
print(PROCESSED_DIR / "test_FD001_features.csv")
print(PROCESSED_DIR / "feature_metadata.json")

Saved:
C:\Users\ahzam\Desktop\Projects\JetGuard\data\processed\train_FD001_features.csv
C:\Users\ahzam\Desktop\Projects\JetGuard\data\processed\validation_FD001_features.csv
C:\Users\ahzam\Desktop\Projects\JetGuard\data\processed\test_FD001_features.csv
C:\Users\ahzam\Desktop\Projects\JetGuard\data\processed\feature_metadata.json


In [9]:
print("Training feature columns:", len(train_features_scaled.columns))
print("Training feature rows:", len(train_features_scaled))
print("Validation feature rows:", len(validation_features_scaled))
print("Test feature rows:", len(test_features_scaled))
train_features_scaled.head()

Training feature columns: 163
Training feature rows: 14459
Validation feature rows: 3272
Test feature rows: 100


,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_8_lag_10,sensor_9_lag_10,sensor_11_lag_10,sensor_12_lag_10,sensor_13_lag_10,sensor_14_lag_10,sensor_15_lag_10,sensor_17_lag_10,sensor_20_lag_10,sensor_21_lag_10
29,1,30,-0.991202,-0.019024,0.0,518.67,-1.057369,0.394226,-1.535615,14.62,...,-0.679959,-1.019477,-1.291665,0.941342,-1.125569,-0.847711,-0.571473,-0.805748,1.248449,1.293668
30,1,31,0.653856,1.691639,0.0,518.67,-1.417198,-1.129636,-1.444690,14.62,...,0.245986,-0.894068,-1.583280,1.465088,-0.211094,-0.570580,-1.041602,-0.805748,1.609710,0.165720
31,1,32,0.242591,-1.045423,0.0,518.67,-0.797492,0.045075,-1.061028,14.62,...,-0.834283,-0.166136,-1.208346,0.836593,-0.973157,-0.802701,-1.085403,-0.805748,0.586137,0.862245
32,1,33,-1.905123,-1.387556,0.0,518.67,-0.037852,-0.441125,-0.831497,14.62,...,-0.679959,-0.998297,-1.166687,0.118314,-0.668332,-0.964093,-0.483871,-0.805748,0.706557,1.638402
33,1,34,0.699552,-0.361157,0.0,518.67,-0.377691,-1.571785,-1.047722,14.62,...,0.400310,-1.138755,-0.375159,1.031127,-0.973157,-0.579581,-1.502972,-0.805748,1.067818,0.765478
